In [ ]:
# --- Tutorial setup ---
# This cell loads everything needed to run this notebook standalone.
# It is hidden in the rendered documentation but visible when running the
# notebook as a tutorial.
import os
import sys
sys.path.insert(0, "/".join(["..", "python"]))

import warnings
import numpy as np
import rasterio
warnings.filterwarnings("ignore", category=rasterio.errors.NotGeoreferencedWarning)

from gridr.chain.grid_resampling_chain import basic_grid_resampling_chain
from gridr.misc.mandrill import mandrill
from notebook_utils import plot_im
from chain_notebook_utils import (
    write_array, create_grid, create_star_polygon, plot_grid_on_image,
)

IN_DOC_BUILD = os.environ.get("DOC_BUILD", "0") == "1"
if not IN_DOC_BUILD:
    from bokeh.io import output_notebook
    output_notebook()

# File paths shared across the chain tutorial series.
RASTER_IN          = "./grid_resampling_chain_raster_in.tif"
GRID_IN_F64        = "./grid_resampling_chain_grid_in_f64.tif"
output_raster_path = "./grid_resampling_chain_output_raster.tif"
output_mask_path   = "./grid_resampling_chain_output_mask.tif"

# Grid parameters shared across the chain tutorial series.
nrow, ncol = 50, 40
origin_pos  = np.array((0.3, 0.2))
origin_node = np.array((0., 0.))
v_row_y, v_row_x = 5.2, 1.2
v_col_y, v_col_x = -2.7, 7.1

grid_row_f64, grid_col_f64 = create_grid(
    nrow, ncol, origin_pos, origin_node,
    v_row_y, v_row_x, v_col_y, v_col_x, dtype=np.float64,
)

# Hybrid input creation: ensure the source raster and grid file exist on disk.
if not os.path.exists(RASTER_IN):
    write_array(mandrill, dtype=mandrill.dtype, fileout=RASTER_IN)
if not os.path.exists(GRID_IN_F64):
    write_array(np.array([grid_row_f64, grid_col_f64]),
                dtype=np.float64, fileout=GRID_IN_F64)

# Default output-dataset open arguments (resolution-dependent overrides
# are applied per-notebook below).
grid_resolution = (8, 8)
from gridr.core.grid.grid_commons import grid_full_resolution_shape
output_shape = grid_full_resolution_shape(
    shape=grid_row_f64.shape, resolution=grid_resolution,
)
raster_out_open_args = {
    "driver": "GTiff", "dtype": np.float64,
    "height": output_shape[0], "width": output_shape[1], "count": 1,
}
mask_out_open_args = {
    "driver": "GTiff", "dtype": np.uint8,
    "height": output_shape[0], "width": output_shape[1],
    "count": 1, "nbits": 1,
}

grid_mask_in_path        = "./grid_resampling_chain_grid_mask_in_u8_1_0.tif"
grid_mask_in_path_i8     = "./grid_resampling_chain_grid_mask_in_i8_0_m10.tif"
raster_mask_in_path_u8   = "./grid_resampling_chain_raster_mask_in_u8_1_0.tif"


# Requesting an output validity mask

This notebook shows how to obtain a binary validity mask for the resampled output. Output pixels are flagged invalid when the resampling could not produce a meaningful value (out-of-domain grid nodes, masked input pixels, etc.). Setting `nodata_out` lets you detect them in the image; an explicit mask makes the validity check unambiguous.

**What you'll learn**

- How to open a dedicated dataset for the output mask
- How to pass it to `basic_grid_resampling_chain` through `mask_out_ds`
- The GridR mask convention (`1 = valid`, `0 = invalid`)
- The 1-bit GeoTIFF storage option

## Setting things up

The setup cell prepares the source raster and the grid file.

We build on top of the `(8, 8)` zoom case from the *Getting Started* notebook.

## Adding a mask output dataset

Pass an opened writable dataset through `mask_out_ds`. GridR's convention is `1 = valid` / `0 = invalid`. To minimise disk usage, the mask is stored as a 1-bit GeoTIFF (`nbits=1` GDAL option).

In [ ]:
with rasterio.open(GRID_IN_F64, "r") as grid_in_ds, \
        rasterio.open(RASTER_IN, "r") as array_src_ds, \
        rasterio.open(output_raster_path, "w", **raster_out_open_args) as array_out_ds, \
        rasterio.open(output_mask_path, "w", **mask_out_open_args) as mask_out_ds:

    basic_grid_resampling_chain(
        grid_ds              = grid_in_ds,
        grid_row_coords_band = 1,
        grid_col_coords_band = 2,
        grid_resolution      = grid_resolution,
        array_src_ds         = array_src_ds,
        array_src_bands      = 1,
        array_out_ds         = array_out_ds,
        interp               = "cubic",
        nodata_out           = 0,
        mask_out_ds          = mask_out_ds,    # <= dedicated mask dataset
    )

In [ ]:
with rasterio.open(output_raster_path, "r") as raster_ds, \
        rasterio.open(output_mask_path, "r") as mask_ds:
    plot_im({"output image": raster_ds.read(1), "output mask": mask_ds.read(1)},
            prefix="002_mask_out")

The output mask makes it easy to filter or composite the resampled image downstream. Subsequent notebooks add input-side masks (grid mask, source raster mask, geometry masks) that all propagate to this output mask.